# 📖 Diário de Bordo: Sistema de Gestão de Estoque (Gest-Stock)
**Data:** Maio de 2026
**Assunto:** Conclusão da Implementação das Fases 2 a 8 (Produtos, Vendas e Dashboard)
**Arquitetura:** Hexagonal com Flask, SQLAlchemy e JWT

Fala equipe! 👋

Nesta última sprint, avançamos significativamente na construção do nosso backend. Finalizamos toda a base necessária para que os mini mercados (Sellers) possam gerenciar seus produtos, realizar vendas e acompanhar as métricas em um dashboard. A arquitetura hexagonal foi rigorosamente mantida para garantir o desacoplamento.

Abaixo está o resumo técnico de tudo o que foi construído e integrado ao nosso repositório, acompanhado dos códigos-fonte em células executáveis/ilustrativas para referência direta da equipe.

### 🗄️ Fase 2: Modelos ORM (Banco de Dados)
Criamos a camada de persistência para as entidades de Produto e Venda, mapeando as tabelas no banco de dados.
* **`product.py`**: Criada a tabela `products` com campos como `name`, `price`, `quantity`, `image_url` e `status`.
* **`sale.py`**: Criada a tabela `sales` vinculada ao produto e ao vendedor.
* **Atualização do Banco (`data_base.py`)**: Registramos os novos modelos para que o SQLAlchemy crie as tabelas automaticamente via `db.create_all()`.

In [ ]:
# src/Infrastructure/Model/product.py
from datetime import datetime
from src.config.data_base import db

class Product(db.Model):
    __tablename__ = 'products'

    id = db.Column(db.Integer, primary_key=True)
    seller_id = db.Column(db.Integer, db.ForeignKey('users.id'), nullable=False)
    name = db.Column(db.String(150), nullable=False)
    price = db.Column(db.Float, nullable=False)
    quantity = db.Column(db.Integer, default=0)
    image_url = db.Column(db.String(300))
    status = db.Column(db.String(20), default='ACTIVE')
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    def to_dict(self):
        return {
            "id": self.id,
            "seller_id": self.seller_id,
            "name": self.name,
            "price": self.price,
            "quantity": self.quantity,
            "image_url": self.image_url,
            "status": self.status,
            "created_at": self.created_at.isoformat() if self.created_at else None,
        }

In [ ]:
# src/Infrastructure/Model/sale.py
from datetime import datetime
from src.config.data_base import db

class Sale(db.Model):
    __tablename__ = 'sales'

    id = db.Column(db.Integer, primary_key=True)
    product_id = db.Column(db.Integer, db.ForeignKey('products.id'), nullable=False)
    seller_id = db.Column(db.Integer, db.ForeignKey('users.id'), nullable=False)
    quantity = db.Column(db.Integer, nullable=False)
    unit_price = db.Column(db.Float, nullable=False)
    total_price = db.Column(db.Float, nullable=False)
    sale_date = db.Column(db.DateTime, default=datetime.utcnow)

    def to_dict(self):
        return {
            "id": self.id,
            "product_id": self.product_id,
            "seller_id": self.seller_id,
            "quantity": self.quantity,
            "unit_price": self.unit_price,
            "total_price": self.total_price,
            "sale_date": self.sale_date.isoformat() if self.sale_date else None,
        }

In [ ]:
# src/config/data_base.py (Trecho Atualizado)
# Adicionado no final de init_db para garantir a criação das tabelas

    from src.Infrastructure.Model import user  # noqa: F401
    from src.Infrastructure.Model import product  # noqa: F401
    from src.Infrastructure.Model import sale  # noqa: F401

    # Cria as tabelas automaticamente
    with app.app_context():
        db.create_all()

### 🔌 Fase 3: Adaptadores de Repositório e Injeção de Dependência
Implementamos os adaptadores concretos que conversam com o banco de dados.
* **`ProductRepositoryAdapter.py` e `SaleRepositoryAdapter.py`**: Implementam as "Ports" da nossa aplicação.
* **`di_container.py`**: Atualizado para injetar essas dependências no projeto.

In [ ]:
# src/Infrastructure/Adapters/ProductRepositoryAdapter.py
from typing import List, Optional
from src.Application.Ports.ProductRepositoryPort import ProductRepositoryPort
from src.Domain.product import ProductDomain, ProductStatus
from src.Infrastructure.Model.product import Product
from src.config.data_base import db

class ProductRepositoryAdapter(ProductRepositoryPort):
    def save(self, p: ProductDomain) -> int:
        product = Product(
            seller_id=p.seller_id, name=p.name, price=p.price,
            quantity=p.quantity, image_url=p.image_url, status=p.status.value,
        )
        db.session.add(product)
        db.session.commit()
        return product.id

    def find_by_seller_id(self, seller_id: int) -> List[ProductDomain]:
        products = Product.query.filter_by(seller_id=seller_id).all()
        return [self._to_domain(p) for p in products]

In [ ]:
# src/Infrastructure/Adapters/SaleRepositoryAdapter.py
from typing import List, Optional
from src.Application.Ports.SaleRepositoryPort import SaleRepositoryPort
from src.Domain.sale import SaleDomain
from src.Infrastructure.Model.sale import Sale
from src.config.data_base import db

class SaleRepositoryAdapter(SaleRepositoryPort):
    def save(self, s: SaleDomain) -> int:
        sale = Sale(
            product_id=s.product_id, seller_id=s.seller_id,
            quantity=s.quantity, unit_price=s.unit_price, total_price=s.total_price,
        )
        db.session.add(sale)
        db.session.commit()
        return sale.id

    def get_total_sales(self, seller_id: int) -> float:
        result = db.session.query(db.func.sum(Sale.total_price)).filter_by(seller_id=seller_id).scalar()
        return float(result) if result else 0.0

In [ ]:
# src/config/di_container.py
from src.Infrastructure.Adapters.UserRepositoryAdapter import UserRepositoryAdapter
from src.Infrastructure.Adapters.ProductRepositoryAdapter import ProductRepositoryAdapter
from src.Infrastructure.Adapters.SaleRepositoryAdapter import SaleRepositoryAdapter
from src.Infrastructure.Adapters.WhatsappAdapter import WhatsappAdapter

class DIContainer:
    _repositories = {}
    _services = {}

    @classmethod
    def setup(cls):
        cls._repositories['user'] = UserRepositoryAdapter()
        cls._repositories['product'] = ProductRepositoryAdapter()
        cls._repositories['sale'] = SaleRepositoryAdapter()
        cls._services['whatsapp'] = WhatsappAdapter()

    @classmethod
    def get_product_repository(cls): return cls._repositories['product']

    @classmethod
    def get_sale_repository(cls): return cls._repositories['sale']

### 🧠 Fases 4 a 8: Use Cases, Controllers, Rotas e Testes
* **Use Cases**: Isolamos toda a lógica core em `src/Application/UseCases/` (CreateProductUseCase, CreateSaleUseCase, GetDashboardStatsUseCase, etc). O caso de venda faz a baixa automática no estoque (`product.decrease_stock(qty)`).
* **Controllers e Rotas**: `product_controller.py`, `sale_controller.py` e `dashboard_controller.py` foram criados e injetados no `routes.py`. Todos protegidos por `@jwt_required()`.
* **Testes**: Suítes em `test_domains.py` (unitários) e `test_integration.py` (ponta a ponta com Flask Test Client e banco em memória).

**Próximos Passos recomendados para a equipe:**
1. Rodem os testes em suas máquinas (`python -m pytest test_integration.py -v`).
2. O Frontend (React) já pode começar a consumir essas novas rotas (`/api/products`, `/api/sales`).
3. Dúvidas sobre o fluxo JWT ou Twilio? Usem o script `test_fluxo_vendedor.py` na raiz para simular a criação/ativação de um usuário.